# UAE IA Compliance Mapping Pipeline — Google Colab

**Runtime:** GPU → T4 (free tier) ~25–40 min | CPU → ~3–4 hrs  
**Steps:**
1. Clone repo from GitHub
2. Mount Google Drive and copy your data files in
3. Install dependencies
4. Run the pipeline
5. Download results back to Drive

**Change runtime to GPU first:**  
`Runtime → Change runtime type → T4 GPU`

## Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Running on CPU — will be slower (~3-4 hrs). Consider enabling GPU.')

## Step 2 — Clone the repo

In [ ]:
import os
from getpass import getpass

REPO_OWNER = 'puneetha08nr'
REPO_NAME  = 'regulatory_parsing_improved'
REPO_DIR   = f'/content/{REPO_NAME}'

# ── Authentication ────────────────────────────────────────────────────────
# If the repo is PUBLIC  → leave TOKEN blank (just press Enter)
# If the repo is PRIVATE → paste your GitHub Personal Access Token
#   Create one at: GitHub → Settings → Developer settings → Personal access tokens
#   Scopes needed: repo (read)
TOKEN = getpass('GitHub token (press Enter if repo is public): ').strip()

if TOKEN:
    REPO_URL = f'https://{TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('Using token authentication.')
else:
    REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('No token — assuming public repo.')

# ── Clone or pull ─────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_NAME}...')
    ret = os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')
    if ret != 0:
        raise RuntimeError(
            'git clone failed.\n'
            '  • If the repo is private, re-run this cell and enter a GitHub token.\n'
            '  • Create a token at: GitHub → Settings → Developer settings → '
            'Personal access tokens → Generate new token (repo scope).'
        )
else:
    print('Repo already cloned — pulling latest changes')
    os.system(f'git -C "{REPO_DIR}" pull')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## Step 3 — Mount Google Drive and copy data files

**Before running this cell**, upload these files to your Google Drive under a folder called `compliance_pipeline_data/`:

| File | Where it lives locally |
|------|------------------------|
| `policies/` folder (all `*_for_mapping.json`) | `data/02_processed/policies/` |
| `uae_ia_controls_structured.json` | `data/02_processed/` |
| `golden_mapping_dataset.json` | `data/07_golden_mapping/` |
| `not_applicable_passages.json` | `data/07_golden_mapping/` |
| `obligation-classifier-legalbert/` folder | repo root |

You can zip the `data/` folder and unzip it here, or upload files individually.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, pathlib, subprocess, os

# ── Set this to the folder name you created in Google Drive ───────────────
DRIVE_DATA = '/content/drive/MyDrive/compliance_pipeline_data'

def copy_from_drive(src, dst, required=False):
    src, dst = pathlib.Path(src), pathlib.Path(dst)
    if not src.exists():
        msg = f'  {"❌ MISSING (required)" if required else "SKIP (optional, not found in Drive)"}: {src.name}'
        print(msg)
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'  ✅ {src.name}')
    return True

print('Copying data from Google Drive...')

# Required: policy passages
copy_from_drive(f'{DRIVE_DATA}/policies',
                f'{REPO_DIR}/data/02_processed/policies', required=True)

# Required: UAE IA controls
copy_from_drive(f'{DRIVE_DATA}/uae_ia_controls_structured.json',
                f'{REPO_DIR}/data/02_processed/uae_ia_controls_structured.json', required=True)

# Optional but recommended: golden dataset (blocklist)
copy_from_drive(f'{DRIVE_DATA}/golden_mapping_dataset.json',
                f'{REPO_DIR}/data/07_golden_mapping/golden_mapping_dataset.json')
copy_from_drive(f'{DRIVE_DATA}/not_applicable_passages.json',
                f'{REPO_DIR}/data/07_golden_mapping/not_applicable_passages.json')

# Optional: LegalBERT model — accepts either a folder OR a zip file
model_dst = pathlib.Path(f'{REPO_DIR}/obligation-classifier-legalbert')
model_folder = pathlib.Path(f'{DRIVE_DATA}/obligation-classifier-legalbert')
model_zip    = pathlib.Path(f'{DRIVE_DATA}/obligation-classifier-legalbert.zip')

if model_folder.exists():
    copy_from_drive(str(model_folder), str(model_dst))
elif model_zip.exists():
    print(f'  Unzipping obligation-classifier-legalbert.zip ...')
    import zipfile
    try:
        with zipfile.ZipFile(str(model_zip), 'r') as zf:
            # Show top-level entries so we know the zip structure
            names = zf.namelist()
            print(f'    zip contains {len(names)} files, e.g.: {names[:3]}')
            zf.extractall(REPO_DIR)
        # The zip may extract to a differently-named folder; rename if needed
        extracted = [pathlib.Path(REPO_DIR) / n.split('/')[0] for n in names if n.strip()]
        extracted_root = extracted[0] if extracted else None
        if extracted_root and extracted_root.exists() and extracted_root != model_dst:
            extracted_root.rename(model_dst)
            print(f'    renamed {extracted_root.name} → obligation-classifier-legalbert')
        print('  ✅ obligation-classifier-legalbert (from zip)')
    except zipfile.BadZipFile as e:
        print(f'  ❌ Bad zip file: {e}')
        print('     Re-zip the folder locally:  zip -r obligation-classifier-legalbert.zip obligation-classifier-legalbert/')
        print('     Then re-upload to Drive and re-run this cell.')
else:
    print('  SKIP: obligation-classifier-legalbert not found — rule-based classifier will be used')

# Verify required directories exist
print()
ok = True
for check in ['data/02_processed/policies', 'data/02_processed/uae_ia_controls_structured.json']:
    p = pathlib.Path(REPO_DIR) / check
    if not p.exists():
        print(f'  ❌ Still missing: {check}  ← upload to Drive and re-run this cell')
        ok = False
if ok:
    print('✅ All required files in place. Proceed to next step.')

## Step 3b — Audit policy files (deduplication check)

Detects mislabelled / duplicate files that silently hurt retrieval quality.  
If it reports missing files, upload them to Drive and re-run Step 3.

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', 'scripts/deduplicate_policies.py', '--check'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 4 — Install dependencies

In [ ]:
# Core pipeline dependencies (docling/flask/label-studio not needed for pipeline run)
!pip install -q \
    transformers \
    sentence-transformers \
    rank-bm25 \
    pandas \
    scikit-learn \
    numpy \
    sentencepiece

print('Install complete.')

## Step 5 — Run the pipeline

In [ ]:
import os
os.chdir(REPO_DIR)

# ── GPU pipeline settings ──────────────────────────────────────────────────
os.environ['CPU_MODE']       = '0'            # disable CPU-lite mode (we have GPU)
os.environ['RERANKER_MODEL'] = 'BAAI/bge-reranker-base'  # best available reranker

# Retrieval depth — these directly control Recall@K
# top_k_retrieve : BM25+Dense candidates per control per doc (first BM25 stage)
# top_k_rerank   : how many pass to cross-encoder            (second CE stage)
# top_k_per_doc  : cross-encoder keeps this many per doc
# top_k           : final passages kept per control (across all docs)
#
# Higher TOP_K_RETRIEVE = higher Recall@K (costs more CE calls, fine on GPU)
# Rule of thumb: TOP_K_RETRIEVE >= TOP_K_RERANK >= TOP_K_PER_DOC
os.environ['TOP_K_RETRIEVE'] = '50'   # retrieve 50 candidates per control per doc
os.environ['TOP_K_RERANK']   = '100'  # pass all 50 to CE (can exceed retrieve if graph used)
os.environ['TOP_K_PER_DOC']  = '5'   # keep top-5 per document after CE
os.environ['TOP_K']          = '10'   # keep top-10 per control overall

# Reranker score thresholds
# BGE-reranker-base (untuned) produces genuine matches in 0.3–0.6 range.
# Lower thresholds until the model is fine-tuned on golden data.
os.environ['THRESHOLD_FULL']    = '0.45'   # was 0.60 — 'Fully Addressed'
os.environ['THRESHOLD_PARTIAL'] = '0.25'   # was 0.35 — 'Partially Addressed'

print("Pipeline settings:")
for k in ['CPU_MODE','RERANKER_MODEL','TOP_K_RETRIEVE','TOP_K_RERANK','TOP_K_PER_DOC','TOP_K',
          'THRESHOLD_FULL','THRESHOLD_PARTIAL']:
    print(f"  {k} = {os.environ[k]}")

# Run the pipeline — output files go to data/06_compliance_mappings/
!python3 quick_start_compliance.py

## Step 6 — Evaluate results (Precision / Recall@K / RePASs)

In [ ]:
os.chdir(REPO_DIR)
!python3 scripts/evaluate_pipeline.py

## Step 7 — Save results back to Google Drive

In [ ]:
import shutil, pathlib, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RESULTS_DIR = f'/content/drive/MyDrive/compliance_pipeline_data/results_{timestamp}'
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

FILES_TO_SAVE = [
    ('data/06_compliance_mappings/mappings.csv',               'mappings.csv'),
    ('data/06_compliance_mappings/mappings.json',              'mappings.json'),
    ('data/06_compliance_mappings/mappings_by_passage.json',   'mappings_by_passage.json'),
    ('data/06_compliance_mappings/compliance_report.json',     'compliance_report.json'),
    ('data/06_compliance_mappings/evaluation_report.json',     'evaluation_report.json'),
    ('data/06_compliance_mappings/retrieval_log.json',         'retrieval_log.json'),
]

for src, dst_name in FILES_TO_SAVE:
    src_path = pathlib.Path(REPO_DIR) / src
    if src_path.exists():
        shutil.copy2(src_path, f'{RESULTS_DIR}/{dst_name}')
        print(f'  ✅ saved: {dst_name}')
    else:
        print(f'  ⚠️  not found: {src}')

# Save per-policy folder
by_policy_src = pathlib.Path(REPO_DIR) / 'data/06_compliance_mappings/by_policy'
if by_policy_src.exists():
    shutil.copytree(by_policy_src, f'{RESULTS_DIR}/by_policy')
    print(f'  ✅ saved: by_policy/')

print(f'\nAll results saved to Google Drive: {RESULTS_DIR}')

## Step 8 — Push results back to GitHub

After this, just run `git pull` on your local machine and the output files appear automatically — no manual download needed.

In [ ]:
import os, subprocess
from getpass import getpass

os.chdir(REPO_DIR)

# Use the same token entered in Step 2 (re-enter if you restarted runtime)
GIT_TOKEN = getpass('GitHub token (same one used in Step 2): ').strip()
GIT_USER  = 'puneetha08nr'
GIT_EMAIL = 'puneetha@example.com'   # can be anything — just for the commit

# Configure git identity for this Colab session
subprocess.run(['git', 'config', 'user.name',  GIT_USER],  cwd=REPO_DIR)
subprocess.run(['git', 'config', 'user.email', GIT_EMAIL], cwd=REPO_DIR)

# Update remote URL to include token so push works without interactive prompt
remote_url = f'https://{GIT_TOKEN}@github.com/{GIT_USER}/regulatory_parsing_improved.git'
subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)

# Stage only the output data files (not the whole repo)
FILES_TO_COMMIT = [
    'data/06_compliance_mappings/mappings.csv',
    'data/06_compliance_mappings/mappings.json',
    'data/06_compliance_mappings/mappings_by_passage.json',
    'data/06_compliance_mappings/compliance_report.json',
    'data/06_compliance_mappings/evaluation_report.json',
    'data/06_compliance_mappings/retrieval_log.json',
    'data/06_compliance_mappings/by_policy',
]

import pathlib
for f in FILES_TO_COMMIT:
    p = pathlib.Path(REPO_DIR) / f
    if p.exists():
        subprocess.run(['git', 'add', str(p)], cwd=REPO_DIR)
        print(f'  staged: {f}')
    else:
        print(f'  skip (not found): {f}')

import datetime
run_ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
result = subprocess.run(
    ['git', 'commit', '-m', f'colab pipeline run: {run_ts}'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

push = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    cwd=REPO_DIR, capture_output=True, text=True
)
if push.returncode == 0:
    print('\n✅ Results pushed to GitHub.')
    print('   On your local machine, run:  git pull')
    print('   → data/06_compliance_mappings/ will be updated automatically.')
else:
    print('❌ Push failed:', push.stderr)
    print('   Check that your token has "repo" write access.')

## (Optional) Step 9 — Generate next annotation batch

Creates a fresh `golden_set_mapping_tasks.json` from the new pipeline output.  
Import this into Label Studio for the next annotation round.

In [ ]:
os.chdir(REPO_DIR)
!python3 create_golden_set_tasks.py \
    --candidates data/06_compliance_mappings/mappings.json \
    --output-tasks data/03_label_studio_input/golden_set_mapping_tasks.json

# Stage and push this file too
subprocess.run(['git', 'add', f'{REPO_DIR}/data/03_label_studio_input/golden_set_mapping_tasks.json'], cwd=REPO_DIR)
result = subprocess.run(['git', 'commit', '-m', 'colab: add annotation tasks for next round'],
                        cwd=REPO_DIR, capture_output=True, text=True)
subprocess.run(['git', 'push', 'origin', 'main'], cwd=REPO_DIR)
print('golden_set_mapping_tasks.json pushed to GitHub → git pull locally to get it')